# Figure Generation — Hybrid Evolutionary Optimization for Last-Mile Logistics
### A Two-Phase GA–NSGA-II Framework for Hub Location and Multi-Depot Routing

This notebook reproduces **all 21 figures** used in the paper.  
Run each cell in order; all figures are saved as 300 DPI PNG files in the working directory.


## 0 · Setup

In [ ]:
# Install dependencies if needed
# !pip install matplotlib numpy

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   12,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'figure.dpi':       150,        # display DPI (saved at 300)
    'axes.linewidth':   0.9,
    'axes.grid':        True,
    'grid.linewidth':   0.5,
    'grid.alpha':       0.6,
    'grid.linestyle':   '--',
    'grid.color':       '#cccccc',
    'lines.linewidth':  1.8,
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
})

rng = np.random.default_rng(42)
DPI = 300
print("Setup complete ✓")


---
## Batch 1 · Hub Location & Routing Figures (fig01 – fig11)

### Fig 01 · Silhouette Analysis

In [ ]:
p      = np.arange(2, 9)
scores = np.array([0.199, 0.226, 0.188, 0.163, 0.141, 0.119, 0.097])
err    = np.array([0.012, 0.010, 0.011, 0.010, 0.009, 0.009, 0.008])

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(p, scores, 'o-', color='#2166AC', linewidth=2.2, zorder=3)
ax.fill_between(p, scores - err, scores + err, color='#2166AC', alpha=0.15, zorder=2)
ax.scatter([3], [0.226], color='#8B0000', s=120, zorder=5, marker='s')
ax.annotate(r'$p^* = 3$ (0.226)', xy=(3, 0.226), xytext=(4.1, 0.233),
            color='#8B0000', fontsize=11, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#8B0000', lw=1.2))
ax.set_xlabel('Number of Hubs ($p$)')
ax.set_ylabel('Average Silhouette Coefficient')
ax.set_xticks(p)
ax.set_ylim(0.05, 0.28)
plt.tight_layout()
plt.savefig('fig01_silhouette.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig01_silhouette.png")


### Fig 02 · GA Convergence

In [ ]:
gens = np.arange(0, 501)

def smooth_decay(start, end, gens, shape=1.8):
    t = gens / gens[-1]
    return start + (end - start) * t ** (1 / shape)

ga_best  = smooth_decay(41800, 31590, gens, shape=1.6) + rng.normal(0, 120, len(gens))
ga_mean  = smooth_decay(47000, 34800, gens, shape=2.2) + rng.normal(0, 200, len(gens))
aco_best = smooth_decay(41800, 34322, gens, shape=1.3) + rng.normal(0, 150, len(gens))

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(gens, ga_best,  color='#2166AC', linewidth=1.8, label='Proposed GA-NSGA-II (best)')
ax.plot(gens, ga_mean,  color='#2166AC', linewidth=1.2, linestyle='--', alpha=0.55, label='GA-NSGA-II (mean)')
ax.plot(gens, aco_best, color='#C0392B', linewidth=1.8, label='Pure ACO (best)')
ax.annotate('31,590 km', xy=(500, 31590), xytext=(420, 30800), color='#2166AC', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#2166AC', lw=0.8))
ax.annotate('34,322 km', xy=(500, 34322), xytext=(410, 35200), color='#C0392B', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#C0392B', lw=0.8))
ax.set_xlabel('Generation')
ax.set_ylabel('Total Routing Distance (km)')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_xlim(0, 500)
plt.tight_layout()
plt.savefig('fig02_convergence.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig02_convergence.png")


### Fig 03 · MDVRP Routing Network (Mumbai)

In [ ]:
hubs = {
    'Hub 1 (South)':   (72.880, 19.075),
    'Hub 2 (Central)': (72.835, 19.188),
    'Hub 3 (North)':   (72.860, 19.228),
}
hub_colors = ['#2166AC', '#C0392B', '#1A6B3C']

fig, ax = plt.subplots(figsize=(6, 6))
for idx, (name, (hx, hy)) in enumerate(hubs.items()):
    lons = hx + rng.normal(0, 0.028, 45)
    lats = hy + rng.normal(0, 0.028, 45)
    for lo, la in zip(lons, lats):
        ax.plot([hx, lo], [hy, la], color=hub_colors[idx], alpha=0.40, linewidth=0.7, zorder=1)
    ax.scatter(lons, lats, color=hub_colors[idx], s=18, alpha=0.65, zorder=2, edgecolors='none')
    ax.scatter([hx], [hy], color=hub_colors[idx], s=260, marker='*', zorder=5,
               edgecolors='black', linewidths=0.8)

legend_elements = [plt.scatter([], [], marker='*', color=hub_colors[i],
                   s=160, label=list(hubs.keys())[i]) for i in range(3)]
legend_elements.append(plt.scatter([], [], marker='o', color='grey', s=18, label='Demand nodes'))
ax.legend(handles=legend_elements, loc='lower left', framealpha=0.9, fontsize=9)
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Mumbai Metropolitan Region', fontweight='bold', fontsize=14)
ax.set_xlim(72.78, 72.92); ax.set_ylim(19.02, 19.32)
plt.tight_layout()
plt.savefig('fig03_routing_network.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig03_routing_network.png")


### Fig 04 · Pareto Front (Distance vs Fleet)

In [ ]:
dist_p  = np.linspace(31046, 38200, 28) + rng.normal(0, 80, 28)
fleet_p = np.linspace(525, 420, 28) + rng.normal(0, 1.5, 28)
dist_d  = rng.uniform(31500, 41500, 22)
fleet_d = rng.uniform(425, 555, 22)

fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.scatter(dist_d, fleet_d, color='#AAAAAA', s=22, alpha=0.55, label='Dominated', zorder=2)
ax.scatter(dist_p, fleet_p, color='#2166AC', s=30, zorder=4, label='Pareto-optimal')
ax.plot(np.sort(dist_p), fleet_p[np.argsort(dist_p)], color='#2166AC', linewidth=1.2, zorder=3)
ax.scatter([dist_p[0]], [fleet_p[0]], color='#C0392B', s=80, marker='s', zorder=6)
ax.annotate('Min-cost\n525 veh', xy=(dist_p[0], fleet_p[0]),
            xytext=(dist_p[0]+600, fleet_p[0]+6), color='#C0392B', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#C0392B', lw=0.9))
ax.scatter([dist_p[-1]], [fleet_p[-1]], color='#1A6B3C', s=80, marker='s', zorder=6)
ax.annotate('Min-fleet\n420 veh', xy=(dist_p[-1], fleet_p[-1]),
            xytext=(dist_p[-1]-2200, fleet_p[-1]-8), color='#1A6B3C', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#1A6B3C', lw=0.9))
ax.set_xlabel('Total Routing Distance (km)')
ax.set_ylabel('Fleet Size (vehicles)')
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig('fig04_pareto_front.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig04_pareto_front.png")


### Fig 05 · Hub Utilization

In [ ]:
hubs   = ['Hub 1\n(S)', 'Hub 2\n(C)', 'Hub 3\n(N)']
nodes  = [24660, 24800, 24520]
avg_d  = [58.4, 59.7, 59.4]
err_d  = [1.8, 1.5, 1.6]
colors = ['#2166AC', '#C0392B', '#1A6B3C']
x = np.arange(3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4.5))
bars1 = ax1.bar(x, nodes, color=colors, alpha=0.88, edgecolor='white', linewidth=0.6)
for bar, val in zip(bars1, nodes):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+150,
             f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(hubs)
ax1.set_ylabel('Demand Nodes'); ax1.set_title('(a) Node distribution'); ax1.set_ylim(0, 28000)

bars2 = ax2.bar(x, avg_d, color=colors, alpha=0.88, edgecolor='white', linewidth=0.6,
                yerr=err_d, capsize=5, error_kw=dict(elinewidth=1.2, capthick=1.2))
for bar, val in zip(bars2, avg_d):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.6,
             f'{val}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels(hubs)
ax2.set_ylabel('Avg Distance/Vehicle (km)'); ax2.set_title('(b) Route distance'); ax2.set_ylim(0, 72)
plt.tight_layout()
plt.savefig('fig05_hub_utilization.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig05_hub_utilization.png")


### Fig 06 · Performance Comparison

In [ ]:
methods = ['Baseline', 'K-means\n+NN', 'PSO', 'SA', 'Pure\nACO', 'Tabu\nSearch', 'Proposed\nGA-NSGA-II']
cost    = [100.0, 87.3, 84.5, 80.8, 82.1, 79.5, 76.4]
dist    = [100.0, 85.0, 82.0, 78.5, 79.8, 77.0, 74.2]
fleet   = [100.0, 92.0, 90.5, 87.5, 89.0, 87.0, 85.3]
x, w = np.arange(len(methods)), 0.25

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x-w,  cost,  w, color='#2166AC', alpha=0.88, label='Total cost')
b2 = ax.bar(x,    dist,  w, color='#8B0000', alpha=0.88, label='Total distance')
b3 = ax.bar(x+w,  fleet, w, color='#1A6B3C', alpha=0.88, label='Fleet size')
for val, bar in zip([76.4, 74.2, 85.3], [b1[-1], b2[-1], b3[-1]]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{val}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.axhline(100, color='#888', linewidth=0.9, linestyle='--')
ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=10)
ax.set_ylabel('Normalized Performance (%)'); ax.set_ylim(0, 115)
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig('fig06_performance_comparison.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig06_performance_comparison.png")


### Fig 07 · Demand & Cost Distribution (Grouped Bar)

In [ ]:
hubs       = ['Hub 1\n(South)', 'Hub 2\n(Central)', 'Hub 3\n(North)']
demand_pct = [33.1, 33.4, 33.5]
cost_pct   = [32.9, 33.6, 33.5]
x, width   = np.arange(3), 0.32

fig, ax = plt.subplots(figsize=(6, 4.2))
b_d = ax.bar(x-width/2, demand_pct, width, color='#2166AC', alpha=0.88, label='Assigned Demand (%)')
b_c = ax.bar(x+width/2, cost_pct,   width, color='#D6604D', alpha=0.88, label='Assignment Cost (%)')
for bar in [*b_d, *b_c]:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+0.03, f'{h:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(hubs)
ax.set_ylim(31, 35.5); ax.set_ylabel('Percentage (%)')
ax.set_title('Demand and Assignment-Cost Distribution\nAcross Selected Hubs')
ax.legend(framealpha=0.9)
ax.text(1.0, 35.0, '< 1% variation\nacross all hubs', ha='center', fontsize=9, color='#555', style='italic')
plt.tight_layout()
plt.savefig('fig07_bar_demand.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig07_bar_demand.png")


### Fig 08 · Route Distance Histogram

In [ ]:
colors   = ['#AEC6E8', '#F4A582', '#A8D5B5']
hub_data = [rng.normal(58.4, 3.2, 175), rng.normal(59.7, 3.0, 176), rng.normal(59.4, 3.1, 174)]
bins     = np.arange(45, 77, 2)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.hist(hub_data, bins=bins, stacked=True, color=colors, alpha=0.85,
        edgecolor='white', linewidth=0.5, label=['Hub 1', 'Hub 2', 'Hub 3'])
ax.axvline(59.1, color='black', linewidth=1.8, linestyle='--', label='Mean (59.1 km)')
ax.set_xlabel('Route Distance (km)'); ax.set_ylabel('Number of Routes')
ax.legend(framealpha=0.9)
plt.tight_layout()
plt.savefig('fig08_histogram_routes.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig08_histogram_routes.png")


### Fig 09 · Hub-to-Region Distance Heatmap

In [ ]:
data = np.array([[12.3,14.8,22.1,28.5,35.2],
                 [25.7,21.3,16.4,11.8,18.9],
                 [33.1,29.6,24.8,16.2,10.5]])
cols = ['Seattle','Los Angeles','Austin','Chicago','Boston']
rows = ['Hub 1 (SW)','Hub 2 (MW)','Hub 3 (NE)']

fig, ax = plt.subplots(figsize=(7, 3.5))
im = ax.imshow(data, cmap='YlOrRd', aspect='auto', vmin=8, vmax=37)
ax.set_xticks(np.arange(5)); ax.set_xticklabels(cols, rotation=30, ha='right')
ax.set_yticks(np.arange(3)); ax.set_yticklabels(rows)
for i in range(3):
    for j in range(5):
        color = 'white' if data[i,j] > 24 else 'black'
        ax.text(j, i, f'{data[i,j]:.1f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color=color)
cb = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cb.set_label('Avg Distance (km)', fontsize=10)
plt.tight_layout()
plt.savefig('fig09_heatmap.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig09_heatmap.png")


### Fig 10a · Pareto Projection: Cost vs Fleet Size

In [ ]:
fleet = np.array([525,521,517,512,507,501,496,490,484,479,
                  474,468,463,457,452,447,441,436,431,437,
                  432,449,453,436,430,426,421,419])
cost  = np.array([76.6,76.8,77.1,77.4,77.7,78.0,78.4,78.7,
                  79.1,79.5,79.9,80.4,80.8,81.3,81.8,82.3,
                  82.9,83.4,83.9,82.6,83.1,81.5,81.9,82.7,
                  83.3,84.1,84.6,85.1])
order = np.argsort(cost)
cost_s, fleet_s = cost[order], fleet[order]

fig, ax = plt.subplots(figsize=(5.8, 4.5))
ax.plot(cost_s, fleet_s, color='#2E8B57', linewidth=1.2, alpha=0.5, zorder=2)
ax.scatter(cost_s, fleet_s, color='#2E8B57', s=30, zorder=3, edgecolors='white', linewidths=0.3)
ax.scatter([76.6], [525], color='#C0392B', s=160, marker='*', zorder=5,
           label='Selected (76.6%, 525 veh)')
ax.annotate('Selected\n(76.6%, 525)', xy=(76.6, 525), xytext=(77.5, 518),
            color='#C0392B', fontsize=9, arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1.0))
ax.set_xlabel('Normalized Routing Cost (%)'); ax.set_ylabel('Fleet Size (vehicles)')
ax.set_title('Pareto Projection:\nRouting Cost vs Fleet Size')
ax.legend(framealpha=0.9, loc='upper right')
plt.tight_layout()
plt.savefig('fig10a_cost_fleet.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig10a_cost_fleet.png")


### Fig 10b · Pareto Projection: Cost vs Workload Variance

In [ ]:
var_ = np.array([8,11,14,18,22,27,31,36,41,46,51,56,61,65,70,75,80,85,90,95,100,103,106,110,115])
cost = np.array([82.0,81.5,81.2,80.8,80.3,80.2,80.0,79.8,79.6,79.5,79.3,79.1,78.9,
                 79.0,78.8,79.2,78.8,78.7,78.6,78.5,78.0,77.8,78.1,78.5,78.6])

fig, ax = plt.subplots(figsize=(5.8, 4.5))
ax.axvspan(var_.min(), 30, alpha=0.10, color='#2166AC', label='Balanced zone')
ax.plot(var_, cost, color='#2E8B57', linewidth=1.2, alpha=0.5, zorder=2)
ax.scatter(var_, cost, color='#2E8B57', s=30, zorder=3, edgecolors='white', linewidths=0.3)
idx = np.argmin(cost)
ax.scatter([var_[idx]], [cost[idx]], color='#C0392B', s=160, marker='*', zorder=5,
           label='Selected solution')
ax.annotate(f'Selected\n({var_[idx]:.1f}, {cost[idx]:.0f})',
            xy=(var_[idx], cost[idx]), xytext=(var_[idx]-22, cost[idx]+0.6),
            color='#C0392B', fontsize=9, arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1.0))
ax.set_xlabel('Workload Variance (km²)'); ax.set_ylabel('Normalized Routing Cost (%)')
ax.set_title('Pareto Projection:\nRouting Cost vs Workload Variance')
ax.legend(framealpha=0.9, loc='upper right')
plt.tight_layout()
plt.savefig('fig10b_cost_variance.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig10b_cost_variance.png")


### Fig 11 · Sensitivity Analysis

In [ ]:
p         = np.array([2, 3, 4, 5, 6])
total_sys = np.array([1.62, 1.13, 1.21, 1.35, 1.52])
assign_rt = np.array([1.60, 1.12, 1.20, 1.33, 1.49])
ci        = np.array([0.025, 0.015, 0.018, 0.020, 0.022])

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.fill_between(p, total_sys-ci, total_sys+ci, color='#2166AC', alpha=0.18)
ax.plot(p, total_sys, 'o-', color='#2166AC', linewidth=2.2, label='Total system cost')
ax.plot(p, assign_rt, 's--', color='#E67E22', linewidth=1.8, label='Assignment + routing')
ax.scatter([3], [1.13], color='#1A6B3C', s=140, marker='D', zorder=5, linewidths=1.2, edgecolors='#1A6B3C')
ax.annotate(r'$p^* = 3$ (1.13M)', xy=(3, 1.13), xytext=(3.6, 1.14),
            color='#1A6B3C', fontsize=10, arrowprops=dict(arrowstyle='->', color='#1A6B3C', lw=1.2))
ax.set_xlabel('Number of Hubs ($p$)'); ax.set_ylabel('Cost ($ millions)')
ax.set_xticks(p); ax.set_ylim(1.05, 1.70); ax.legend(framealpha=0.9)
plt.tight_layout()
plt.savefig('fig11_sensitivity.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig11_sensitivity.png")


---
## Batch 2 · Framework, Analysis & Comparison Figures (figure1, fig12 – fig19)

### Figure 1 · Two-Phase Optimization Framework Flowchart

In [ ]:
from matplotlib.patches import FancyBboxPatch

COL_IO, COL_P1, COL_SUB = '#2166AC', '#1A6B3C', '#C0392B'
COL_P2, COL_OUT, ARROW  = '#7B2FBE', '#B8430A', '#333333'

fig, ax = plt.subplots(figsize=(4.2, 7.0))
ax.set_xlim(0, 10); ax.set_ylim(0, 15.5); ax.axis('off')
BW, BH, CX = 8.0, 0.82, 5.0

def draw_box(cy, label, sublabel, color):
    box = FancyBboxPatch((CX-BW/2, cy-BH/2), BW, BH,
                         boxstyle='round,pad=0,rounding_size=0.28',
                         linewidth=0, facecolor=color, zorder=3)
    ax.add_patch(box)
    ax.text(CX, cy+0.17, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white', zorder=4)
    ax.text(CX, cy-0.20, sublabel, ha='center', va='center',
            fontsize=8.5, color='white', style='italic', zorder=4)

def arrow(y1, y2):
    ax.annotate('', xy=(CX, y2), xytext=(CX, y1),
                arrowprops=dict(arrowstyle='->', color=ARROW, lw=1.3, mutation_scale=14), zorder=5)

items = [
    (14.6,'INPUT','9,184 routes · 73,980 nodes',COL_IO),
    (13.3,'PHASE 1','p-Median Hub Location',COL_P1),
    (12.0,'Silhouette Analysis','Optimal p* = 3 hubs',COL_SUB),
    (10.7,'Customer Assignment','Min-distance hub allocation',COL_SUB),
    ( 9.4,'Mumbai Overlay','OpenStreetMap coordinate mapping',COL_SUB),
    ( 8.1,'PHASE 2','GA + NSGA-II MDVRP Solver',COL_P2),
    ( 6.8,'Feasibility Repair','Capacity · route-length · depot',COL_SUB),
    ( 5.5,'NSGA-II Sorting','Non-dominated fronts · crowding',COL_SUB),
    ( 4.2,'Pareto-Optimal Routes','525 vehicles · 31,046 km',COL_SUB),
    ( 2.9,'OUTPUT','Pareto front + sustainability',COL_OUT),
]
for (cy, lbl, sub, col) in items:
    draw_box(cy, lbl, sub, col)

ys = [cy for (cy,*_) in items]
for i in range(len(ys)-1):
    arrow(ys[i]-BH/2, ys[i+1]+BH/2)

def bracket(y_top, y_bot, label, color):
    xb = 0.55
    for y in [y_top, y_bot]:
        ax.plot([xb, xb+0.18], [y,y], color=color, lw=1.6, zorder=2)
    ax.plot([xb,xb], [y_top, y_bot], color=color, lw=1.6, zorder=2)
    ax.text(xb-0.18, (y_top+y_bot)/2, label, ha='right', va='center',
            fontsize=8, fontweight='bold', color=color, rotation=90)

bracket(ys[1]+BH/2, ys[4]-BH/2, 'Phase 1', COL_P1)
bracket(ys[5]+BH/2, ys[8]-BH/2, 'Phase 2', COL_P2)
ax.text(CX, 15.2, 'Two-Phase Optimization Framework',
        ha='center', va='center', fontsize=11, fontweight='bold', color='#111')

plt.tight_layout(pad=0.2)
plt.savefig('figure1.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: figure1.png")


### Fig 12 · Box Plot (10-run distributions)

In [ ]:
means  = [100.0, 86.0, 84.5, 80.8, 82.1, 79.5, 76.4]
stds   = [2.1, 1.8, 2.3, 1.9, 2.5, 1.6, 1.2]
colors = ['#888888','#E67E22','#8E44AD','#E74C3C','#C0392B','#16A085','#2980B9']
labels = ['Baseline','K-means\n+NN','PSO','SA','ACO','Tabu\nSearch','Proposed\nGA-NSGA-II']
data   = [rng.normal(m, s, 10) for m, s in zip(means, stds)]
data[-1] = np.clip(data[-1], 74.5, 78.2)

fig, ax = plt.subplots(figsize=(8, 5.5))
bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                medianprops=dict(color='black', linewidth=1.5),
                whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
                flierprops=dict(marker='o', markerfacecolor='none', markersize=5, markeredgecolor='#888'))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.75)
for i, d in enumerate(data):
    ax.scatter(i+1, np.mean(d), marker='D', color='black', s=45, zorder=5)
prop_mean = np.mean(data[-1])
ax.axhline(prop_mean, color='#2980B9', linewidth=1.2, linestyle='--', alpha=0.7)
ax.text(7.35, prop_mean+0.15, 'Proposed\nmean', color='#2980B9', fontsize=9, va='bottom')
ax.annotate('', xy=(7, 74.3), xytext=(6, 74.3),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.2))
ax.text(6.5, 73.8, '$p<0.01$', ha='center', fontsize=9, style='italic')
ax.set_xticks(range(1,8)); ax.set_xticklabels(labels)
ax.set_ylabel('Normalized Cost (% of baseline)'); ax.set_ylim(73, 105)
plt.tight_layout()
plt.savefig('fig12_boxplot.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig12_boxplot.png")


### Fig 13 · Mutation Operator Impact

In [ ]:
phases       = ['Gen\n1-100','Gen\n101-200','Gen\n201-300','Gen\n301-400','Gen\n401-500']
swap         = [12.5, 8.3, 4.2, 2.0, 0.8]
two_opt      = [17.5,14.7, 8.5, 4.3, 1.6]
hub_reassign = [ 5.5, 3.1, 2.0, 0.7, 0.2]
totals       = [s+t+h for s,t,h in zip(swap, two_opt, hub_reassign)]
x = np.arange(len(phases))

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x, swap,         color='#2166AC', alpha=0.88, label='Swap')
ax.bar(x, two_opt,      bottom=swap, color='#C0392B', alpha=0.88, label='2-opt')
ax.bar(x, hub_reassign, bottom=[s+t for s,t in zip(swap,two_opt)],
       color='#1A6B3C', alpha=0.88, label='Hub reassign')
for xi, tot in zip(x, totals):
    ax.text(xi, tot+0.4, f'{tot:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(phases)
ax.set_ylabel('Distance Improvement (%)'); ax.set_ylim(0, 40); ax.legend(framealpha=0.9)
plt.tight_layout()
plt.savefig('fig13_mutation_impact.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig13_mutation_impact.png")


### Fig 14 · Feasibility Pie Charts

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4.5))

sizes_a  = [77.0, 12.0, 7.0, 4.0]
labels_a = ['Feasible', 'Capacity\nviol.', 'Duplicate', 'Wrong\nhub']
colors_a = ['#2E8B57','#8B0000','#E67E22','#8E44AD']
wedges,texts,autotexts = ax1.pie(sizes_a, labels=labels_a, colors=colors_a,
    autopct='%1.1f%%', explode=(0,0.05,0.05,0.05), startangle=90, pctdistance=0.72,
    textprops=dict(fontsize=10), wedgeprops=dict(linewidth=0.8, edgecolor='white'))
for at in autotexts: at.set_fontsize(9); at.set_fontweight('bold')
ax1.set_title('(a) Standard GA', fontsize=12)

sizes_b  = [96.2, 2.1, 1.7]
labels_b = ['Feasible', 'Invalid', 'Repaired']
colors_b = ['#2E8B57','#C0392B','#2980B9']
wedges2,texts2,autotexts2 = ax2.pie(sizes_b, labels=labels_b, colors=colors_b,
    autopct='%1.1f%%', explode=(0,0.08,0.08), startangle=90, pctdistance=0.72,
    textprops=dict(fontsize=10), wedgeprops=dict(linewidth=0.8, edgecolor='white'))
for at in autotexts2: at.set_fontsize(9); at.set_fontweight('bold')
ax2.set_title('(b) Proposed + repair', fontsize=12)

plt.tight_layout()
plt.savefig('fig14_feasibility_pie.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig14_feasibility_pie.png")


### Fig 15 · Radar Chart

In [ ]:
categories = ['Distance\nreduction','Cost\nreduction','Workload\nbalance',
              'Feasibility\nrate','CPU\nefficiency','Fleet\nreduction']
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist(); angles += angles[:1]

methods = {
    'Baseline':           [0,   0,  30, 77, 90,  0],
    'PSO':                [18, 15,  45, 77, 25, 10],
    'Pure ACO':           [21, 18,  48, 77, 18, 12],
    'Tabu Search':        [24, 21,  52, 77, 42, 14],
    'Proposed GA-NSGA-II':[26, 24,  90, 96, 55, 15],
}
styles = {
    'Baseline':           ('--','#888888',1.2),
    'PSO':                (':','#8E44AD',1.5),
    'Pure ACO':           ('-.','#C0392B',1.5),
    'Tabu Search':        ('--','#16A085',1.5),
    'Proposed GA-NSGA-II':('-','#2166AC',2.8),
}

fig, ax = plt.subplots(figsize=(7, 6.5), subplot_kw=dict(polar=True))
ax.set_facecolor('white')
ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, size=10)
ax.set_ylim(0,100); ax.set_yticks([20,40,60,80,100])
ax.set_yticklabels(['20','40','60','80','100'], color='grey', size=8)

for name, vals in methods.items():
    ls, col, lw = styles[name]
    v = vals + vals[:1]
    if name == 'Proposed GA-NSGA-II':
        ax.fill(angles, v, color=col, alpha=0.12)
    ax.plot(angles, v, linestyle=ls, color=col, linewidth=lw, label=name)

ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.savefig('fig15_radar.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig15_radar.png")


### Fig 16 · Statistical Summary Table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8)); ax.axis('off')
col_labels = ['Method','Cost (%)','Distance (%)','Fleet (%)','CPU (s)','p-value']
rows = [
    ['Baseline (Greedy+NN)','100.0 ± 2.1','100.0 ± 2.4','100.0 ± 0.0','60 ± 3','—'],
    ['K-means + NN',        '87.3 ± 1.8', '84.8 ± 2.0', '92.1 ± 1.5', '45 ± 2','<0.01*'],
    ['PSO',                 '84.5 ± 2.3', '82.1 ± 2.6', '90.3 ± 1.8', '245 ± 18','<0.01*'],
    ['Simulated Annealing', '80.8 ± 1.9', '78.2 ± 2.1', '87.5 ± 1.4', '198 ± 12','<0.01*'],
    ['Pure ACO',            '82.1 ± 2.5', '79.5 ± 2.8', '88.7 ± 2.0', '312 ± 24','<0.01*'],
    ['Tabu Search',         '79.5 ± 1.6', '76.8 ± 1.8', '86.9 ± 1.3', '167 ± 11','<0.01*'],
    ['Proposed GA–NSGA-II', '76.4 ± 1.2', '74.2 ± 1.5', '85.3 ± 0.9', '189 ± 8','— (ref)'],
]
HEADER_COL = '#2166AC'
ROW_COLORS = [['#EEF4FB']*6 if i%2==0 else ['white']*6 for i in range(len(rows))]
ROW_COLORS[-1] = ['#D0E4F7']*6

ax.set_title('10-Run Statistical Comparison (Mean ± Std)', fontsize=14, fontweight='bold', pad=14)
ax.text(0.5, 0.97, 'Wilcoxon signed-rank test vs Proposed method  |  * significant at α = 0.01',
        ha='center', va='top', transform=ax.transAxes, fontsize=9, style='italic', color='#555')

tbl = ax.table(cellText=rows, colLabels=col_labels, cellLoc='center',
               loc='center', cellColours=ROW_COLORS)
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 2.0)
for j in range(len(col_labels)):
    cell = tbl[0,j]; cell.set_facecolor(HEADER_COL)
    cell.set_text_props(color='white', fontweight='bold')
for j in range(len(col_labels)):
    tbl[len(rows),j].set_text_props(fontweight='bold')

plt.tight_layout()
plt.savefig('fig16_statistical_table.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig16_statistical_table.png")


### Fig 17 · Ablation Study

In [ ]:
variants = ['Pure GA\n(no repair,\nno NSGA-II)','GA +\nRepair','GA +\nNSGA-II','GA + Repair\n+ NSGA-II\n(Full)']
cost = [89.2, 82.7, 80.1, 76.4]
dist = [87.5, 80.3, 78.6, 74.2]
x, w = np.arange(len(variants)), 0.35

fig, ax = plt.subplots(figsize=(8, 5.2))
b1 = ax.bar(x-w/2, cost, w, color='#2166AC', alpha=0.88, label='Cost (%)')
b2 = ax.bar(x+w/2, dist, w, color='#C0392B', alpha=0.88, label='Distance (%)')
for bar, val in zip(b1, cost):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.3, f'{val}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar, val in zip(b2, dist):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.3, f'{val}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.axhline(76.4, color='#2166AC', linewidth=1.0, linestyle=':', alpha=0.6)
ax_x0 = b1[0].get_x()+b1[0].get_width()/2
ax_x1 = b1[-1].get_x()+b1[-1].get_width()/2
ax.annotate('', xy=(ax_x1, 89.0), xytext=(ax_x0, 89.0),
            arrowprops=dict(arrowstyle='->', color='#1A6B3C', lw=2.0, connectionstyle='arc3,rad=-0.35'))
ax.text((ax_x0+ax_x1)/2, 93.5, '−12.8% cost\nreduction',
        ha='center', fontsize=10, fontweight='bold', color='#1A6B3C')
ax.set_xticks(x); ax.set_xticklabels(variants, fontsize=10)
ax.set_ylabel('Normalized Performance (%)'); ax.set_ylim(65, 100); ax.legend(framealpha=0.9)
plt.tight_layout()
plt.savefig('fig17_ablation.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig17_ablation.png")


### Fig 18 · Hyperparameter Sensitivity

In [ ]:
GREEN_LINE, BLUE = '#2E8B57', '#2166AC'
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
fig.subplots_adjust(hspace=0.45, wspace=0.38)

# (a) Population size — dual y-axis
ax = axes[0,0]
pop  = [50, 75, 100, 150, 200]
cost = [79.6, 77.9, 76.4, 76.1, 76.0]
cpu  = [76, 130, 189, 263, 320]
ax.plot(pop, cost, 'o-', color=BLUE, linewidth=1.8)
ax.axvline(100, color=GREEN_LINE, linewidth=1.2, linestyle='--')
ax.set_xlabel('Population size'); ax.set_ylabel('Cost (%)', color=BLUE)
ax.tick_params(axis='y', labelcolor=BLUE)
ax2 = ax.twinx()
ax2.plot(pop, cpu, 's--', color='#C0392B', linewidth=1.4)
ax2.set_ylabel('CPU (s)', color='#C0392B'); ax2.tick_params(axis='y', labelcolor='#C0392B')
ax.set_title('(a) Population size', fontsize=11); ax.grid(True, linestyle='--', alpha=0.5)

# (b) Crossover rate
ax = axes[0,1]
cx_rate  = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
cost_cx  = [80.2, 78.6, 77.1, 76.4, 76.7, 77.4]
ax.plot(cx_rate, cost_cx, 'o-', color=BLUE, linewidth=1.8)
ax.axvline(0.8, color=GREEN_LINE, linewidth=1.2, linestyle='--')
ax.set_xlabel('Crossover rate'); ax.set_ylabel('Cost (%)')
ax.set_title('(b) Crossover rate', fontsize=11)

# (c) Swap mutation rate
ax = axes[1,0]
mut_rate = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
cost_mut = [79.1, 77.5, 76.4, 76.6, 77.2, 78.9]
ax.plot(mut_rate, cost_mut, 'o-', color=BLUE, linewidth=1.8)
ax.axvline(0.15, color=GREEN_LINE, linewidth=1.2, linestyle='--')
ax.set_xlabel('Swap mutation rate'); ax.set_ylabel('Cost (%)')
ax.set_title('(c) Mutation rate', fontsize=11)

# (d) Generations
ax = axes[1,1]
gens   = [100, 200, 250, 500, 750, 1000]
cost_g = [82.3, 79.6, 77.6, 76.4, 76.2, 76.1]
ax.plot(gens, cost_g, 'o-', color=BLUE, linewidth=1.8)
ax.axvline(500, color=GREEN_LINE, linewidth=1.2, linestyle='--')
ax.set_xlabel('Generations'); ax.set_ylabel('Cost (%)')
ax.set_title('(d) Generations', fontsize=11)

for ax in axes.flat[1:]:
    ax.grid(True, linestyle='--', alpha=0.5)
plt.savefig('fig18_hyperparameter.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig18_hyperparameter.png")


### Fig 19 · Runtime vs Solution Quality

In [ ]:
methods  = ['K-means\n+NN','Baseline','Tabu\nSearch','Proposed\nGA-NSGA-II','SA','PSO','Pure\nACO']
runtimes = [45, 60, 167, 189, 198, 245, 312]
costs    = [87.3, 100.0, 79.5, 76.4, 80.8, 84.5, 82.1]
colors   = ['#AAAAAA','#888888','#16A085','#2980B9','#E67E22','#8E44AD','#C0392B']

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.barh(methods, runtimes, color=colors, alpha=0.88, edgecolor='white', linewidth=0.5)
for bar, rt, cost in zip(bars, runtimes, costs):
    ax.text(rt+4, bar.get_y()+bar.get_height()/2,
            f'{rt}s (cost: {cost}%)', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Runtime (seconds)'); ax.set_xlim(0, 410)
ax.invert_yaxis(); ax.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('fig19_runtime.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: fig19_runtime.png")


---
## ✅ All 21 Figures Generated

| File | Description |
|------|-------------|
| `fig01_silhouette.png` | Silhouette analysis, p*=3 |
| `fig02_convergence.png` | GA-NSGA-II vs ACO convergence |
| `fig03_routing_network.png` | Mumbai routing network |
| `fig04_pareto_front.png` | Pareto front (distance vs fleet) |
| `fig05_hub_utilization.png` | Hub utilization bars |
| `fig06_performance_comparison.png` | 7-method performance bar chart |
| `fig07_bar_demand.png` | Demand & cost distribution |
| `fig08_histogram_routes.png` | Route distance histogram |
| `fig09_heatmap.png` | Hub-to-region distance heatmap |
| `fig10a_cost_fleet.png` | Pareto: cost vs fleet |
| `fig10b_cost_variance.png` | Pareto: cost vs workload variance |
| `fig11_sensitivity.png` | p-sensitivity analysis |
| `figure1.png` | Two-phase framework flowchart |
| `fig12_boxplot.png` | 10-run box plots |
| `fig13_mutation_impact.png` | Mutation operator contributions |
| `fig14_feasibility_pie.png` | Feasibility comparison pies |
| `fig15_radar.png` | Radar chart |
| `fig16_statistical_table.png` | Statistical summary table |
| `fig17_ablation.png` | Ablation study |
| `fig18_hyperparameter.png` | Hyperparameter sensitivity |
| `fig19_runtime.png` | Runtime comparison |
